In [1]:
import numpy as np
import urllib.request

In [2]:
# url = ("https://raw.githubusercontent.com/rasbt/"
# "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
# "the-verdict.txt")
# file_path = "the-verdict.txt"
# urllib.request.urlretrieve(url, file_path)

In [3]:
with open("../data/the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:205])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich


In [4]:
import re 

tokens = re.split(r'([.,?!:;"()\']|--|\s)',raw_text)
tokens = [item for item in tokens if item.strip()]
print(len(tokens))
print(tokens[:99])


4654
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter']


In [5]:
all_words = sorted(set(tokens))
all_words.extend(["<|endoftext|>","<|unk|>"])
vocab_size = len(all_words)
print(vocab_size)

1141


In [6]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i , item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry_', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [7]:
import re 

class SimpleTextTokenizerV1:
    def __init__(self,vocab:dict):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        tokens = re.split(r'([.,?!:;"()\']|--|\s)',text)
        tokens = [item for item in tokens if item.strip()]
        ids = [self.str_to_int[token] for token in tokens]
        return ids
    
    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) #Removes whitespace in front of the punctuation
        return text

In [8]:
tokenizer = SimpleTextTokenizerV1(vocab=vocab)

text = """It's the last he painted, you know, 
Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

final_text = tokenizer.decode(ids)
print(final_text)

[56, 2, 859, 997, 612, 544, 756, 5, 1135, 606, 5, 67, 7, 38, 860, 1117, 764, 803, 7]
It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride.


In [9]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1136)
('your', 1137)
('yourself', 1138)
('<|endoftext|>', 1139)
('<|unk|>', 1140)


In [10]:
class SimpleTextTokenizerV2:
    def __init__(self,vocab:dict):
        self.str_to_int = vocab
        self.int_to_str = {s:i for i,s in vocab.items()}

    def encode(self,text):
        tokens = re.split(r'([.,?!:;"()\']|--|\s)',text)
        tokens = [item for item in tokens if item.strip()]
        tokens = [item if item in self.str_to_int else "<|unk|>" for item in tokens]

        ids = [self.str_to_int[s] for s in tokens]
        return ids 

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) #Removes whitespace in front of the punctuation
        return text

In [11]:
my_tokenizer = SimpleTextTokenizerV2(vocab=vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

ids = my_tokenizer.encode(text)
print(ids)

final_text = my_tokenizer.decode(ids)
print(final_text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1140, 5, 368, 1135, 638, 984, 10, 1139, 55, 997, 965, 993, 732, 997, 1140, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [12]:
import tiktoken

tokenier_tik_token = tiktoken.get_encoding("gpt2")
text = "Akwirw ier"

integers = tokenier_tik_token.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[33901, 86, 343, 86, 220, 959]


In [13]:
strings = tokenier_tik_token.decode(integers)
print(strings)

Akwirw ier


In [14]:
import tiktoken

with open("../data/the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

bpe = tiktoken.get_encoding("gpt2")
enc_text  = bpe.encode(raw_text)
print(len(enc_text))

5145


In [15]:
enc_sample = enc_text[50:]

In [16]:
#Generating input-target pairs. 
context_size = 4 #The number of items in the inputs
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x:{x}")
print(f"y:     {y}")


x:[290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [17]:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desire = enc_sample[i]
    print(f"{bpe.decode(context)}: ----> {bpe.decode([desire])}")

 and: ---->  established
 and established: ---->  himself
 and established himself: ---->  in
 and established himself in: ---->  a


In [18]:
from torch.utils.data import Dataset,DataLoader
import torch 

class GPTDatasetV1(Dataset):
    def __init__(self,txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
        
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [19]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
    dataLoader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
    return dataLoader

In [20]:
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print(f"Inputs: \n {inputs}")
print(f"Targets: \n {targets}")


Inputs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [21]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2241,   287,   257,  4489],
        [   64,   319,   262, 34686],
        [41976,    13,   357, 10915]]), tensor([[  262,  6001,   286,   465],
        [13476,    11,   339,   550],
        [ 5710,   465, 12036,    11],
        [ 6405,   257,  5527, 27075],
        [   11,   290,  4920,  2241],
        [  287,   257,  4489,    64],
        [  319,   262, 34686, 41976],
        [   13,   357, 10915,   314]])]


## Excersise 2.2

In [22]:
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=2,stride=2,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print(f"Inputs: \n {inputs}")
print(f"Targets: \n {targets}")


Inputs: 
 tensor([[   40,   367],
        [ 2885,  1464],
        [ 1807,  3619],
        [  402,   271],
        [10899,  2138],
        [  257,  7026],
        [15632,   438],
        [ 2016,   257]])
Targets: 
 tensor([[  367,  2885],
        [ 1464,  1807],
        [ 3619,   402],
        [  271, 10899],
        [ 2138,   257],
        [ 7026, 15632],
        [  438,  2016],
        [  257,   922]])


In [23]:
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=8,stride=2,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print(f"Inputs: \n {inputs}")
print(f"Targets: \n {targets}")


Inputs: 
 tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438],
        [10899,  2138,   257,  7026, 15632,   438,  2016,   257],
        [  257,  7026, 15632,   438,  2016,   257,   922,  5891],
        [15632,   438,  2016,   257,   922,  5891,  1576,   438],
        [ 2016,   257,   922,  5891,  1576,   438,   568,   340]])
Targets: 
 tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
        [ 1464,  1807,  3619,   402,   271, 10899,  2138,   257],
        [ 3619,   402,   271, 10899,  2138,   257,  7026, 15632],
        [  271, 10899,  2138,   257,  7026, 15632,   438,  2016],
        [ 2138,   257,  7026, 15632,   438,  2016,   257,   922],
        [ 7026, 15632,   438,  2016,   257,   922,  5891,  1576],
        [  438,  2016,   257,   922,  5891,  1576,   4

In [24]:
import torch 

torch.manual_seed(123)

vocab_size = 6
ouput_dim = 3 

embedding_layer = torch.nn.Embedding(vocab_size,ouput_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [25]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [26]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)


In [27]:
import tiktoken

max_length = 4
dataloader = create_dataloader_v1(
    txt=raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False
)

data_iter = iter(dataloader)
inputs,targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)



Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [28]:
with open("../data/the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

In [29]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [30]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [31]:
input_embeddings = pos_embeddings + token_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Why are we encoding word positions?

Consider the following 2 sentences:
1)The man bit the cat 
2)The cat bit the man 

These 2 sentences are different. But when we give these sentences to an embedding model embedding for man in the first sentence and the embedding for man in the 2nd sentence will be same. Why is this an issue...? As we can see the place at which the word comes in the sentence plays a really important role!! So we need to give the LLM some context on the position of the words! 

This is done by positional embedding... 

In [32]:
text = "Your journey starts with one step"

ids = tokenier_tik_token.encode(text=text)
words = tokenier_tik_token.decode(ids) 

print(ids)
print(words)


[7120, 7002, 4940, 351, 530, 2239]
Your journey starts with one step


In [36]:
embedding_layer = torch.nn.Embedding(vocab_size,3)
embeddings = embedding_layer(torch.tensor(ids))

In [37]:
embeddings

tensor([[-0.2973, -0.5159,  0.2349],
        [-1.1063,  1.2938,  0.0885],
        [ 0.7876, -0.4010, -0.9707],
        [-0.9514,  0.3072,  0.8707],
        [ 0.0821,  0.4568, -0.1773],
        [-0.1334, -0.2590,  1.9325]], grad_fn=<EmbeddingBackward0>)

In [38]:
query = embeddings[1] 
attention_scores2 = torch.zeros(embeddings.shape[0])

for i,x_i in enumerate(embeddings):
    attention_scores2[i] = torch.dot(query,x_i)

print(attention_scores2)


tensor([-0.3178,  2.9057, -1.4760,  1.5271,  0.4845, -0.0165],
       grad_fn=<CopySlices>)


In [40]:
attention_scores2_temp = attention_scores2 / attention_scores2.sum()
print(attention_scores2_temp)
print(attention_scores2_temp.sum())

tensor([-0.1023,  0.9353, -0.4751,  0.4915,  0.1559, -0.0053],
       grad_fn=<DivBackward0>)
tensor(1.0000, grad_fn=<SumBackward0>)


In [41]:
def soft_max_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

In [42]:
attention_scores2_naive = soft_max_naive(attention_scores2)
print("Attention weights:", attention_scores2_naive)
print("Sum:", attention_scores2_naive.sum())

Attention weights: tensor([0.0275, 0.6912, 0.0086, 0.1741, 0.0614, 0.0372],
       grad_fn=<DivBackward0>)
Sum: tensor(1.0000, grad_fn=<SumBackward0>)


In [43]:
attention_weights2 = torch.softmax(attention_scores2,dim=0)
print("Attention weights:", attention_weights2)
print("Sum:", attention_weights2.sum())

Attention weights: tensor([0.0275, 0.6912, 0.0086, 0.1741, 0.0614, 0.0372],
       grad_fn=<SoftmaxBackward0>)
Sum: tensor(1., grad_fn=<SumBackward0>)


In [ ]:
context_vec2 = torch.zeros(query.shape)
for i,x_i in enumerate(embeddings):
    context_vec2 += attention_weights2[i]*x_i
print(context_vec2)

tensor([-0.9316,  0.9485,  0.2718], grad_fn=<AddBackward0>)


In [59]:
attn_scores = torch.empty(6,6)
for i,x_i in enumerate(embeddings):
    for j,x_j in enumerate(embeddings):
        attn_scores[i,j] = torch.dot(x_i,x_j)

print(attn_scores)

attn_weights = torch.empty(6,6)
for i , x_i in enumerate(attn_scores):
    attn_weights[i] = torch.softmax(x_i,dim=0)
print(attn_weights)

tensor([[ 0.4097, -0.3178, -0.2553,  0.3289, -0.3017,  0.6272],
        [-0.3178,  2.9057, -1.4760,  1.5271,  0.4845, -0.0165],
        [-0.2553, -1.4760,  1.7235, -1.7178,  0.0536, -1.8772],
        [ 0.3289,  1.5271, -1.7178,  1.7577, -0.0922,  1.7300],
        [-0.3017,  0.4845,  0.0536, -0.0922,  0.2468, -0.4719],
        [ 0.6272, -0.0165, -1.8772,  1.7300, -0.4719,  3.8194]],
       grad_fn=<CopySlices>)
tensor([[0.2149, 0.1038, 0.1105, 0.1982, 0.1055, 0.2671],
        [0.0275, 0.6912, 0.0086, 0.1741, 0.0614, 0.0372],
        [0.0969, 0.0286, 0.7010, 0.0224, 0.1320, 0.0191],
        [0.0750, 0.2486, 0.0097, 0.3130, 0.0492, 0.3045],
        [0.1186, 0.2604, 0.1693, 0.1463, 0.2053, 0.1001],
        [0.0341, 0.0179, 0.0028, 0.1028, 0.0114, 0.8309]],
       grad_fn=<CopySlices>)


In [ ]:
context_vec = torch.empty(6,6)
for i,x_i in enumerate(embeddings):
    context_vec_i = torch.zeros(query.shape)
for i,x_i in enumerate(embeddings):
    context_vec2 += attention_weights2[i]*x_i
print(context_vec2)

In [ ]:
a